## Resources


In [ ]:
import os
import json
import logging
import time
from typing import TypedDict, List, Optional, Literal
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

import requests
from pydantic import BaseModel, Field, ValidationError

from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, END

# ===========================================================================
# 1. CONFIG / SECRETS
# ===========================================================================

ANTHROPIC_MODEL = "claude-sonnet-4-6"
RAPIDAPI_KEY = os.environ.get("RAPIDAPI_KEY", "")
RAPIDAPI_HOST = "real-time-amazon-data.p.rapidapi.com"  # swap for whichever Amazon API you subscribe to

MAX_CANDIDATES = 8          # how many products to pull from search
MAX_RETRY_LOOSEN = 1        # how many times we auto-relax budget before giving up
BUDGET_RELAX_PCT = 0.15     # widen budget ceiling by 15% on retry

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("amazon_agent")

llm = ChatAnthropic(model=ANTHROPIC_MODEL, temperature=0, max_tokens=1500)


# ===========================================================================
# 2. STRUCTURED SCHEMAS (Pydantic) — used to force reliable LLM JSON output
# ===========================================================================

class QueryIntent(BaseModel):
    core_query: str = Field(..., description="cleaned search string to send to Amazon search API")
    max_budget_usd: Optional[float] = Field(None, description="explicit or implied upper price bound, null if none stated")
    min_rating: float = Field(4.0, description="minimum acceptable star rating, default 4.0")
    must_have_attributes: List[str] = Field(default_factory=list, description="non-negotiable features extracted from the keyword")
    nice_to_have_attributes: List[str] = Field(default_factory=list)


class Product(BaseModel):
    asin: str
    title: str
    price: Optional[float] = None
    rating: Optional[float] = None
    review_count: Optional[int] = None
    url: Optional[str] = None
    thumbnail: Optional[str] = None
    review_snippets: List[str] = Field(default_factory=list)
    quality_score: Optional[float] = None
    quality_rationale: Optional[str] = None


class QualityAssessment(BaseModel):
    quality_score: float = Field(..., ge=0, le=10)
    rationale: str
    red_flags: List[str] = Field(default_factory=list)


class FinalRecommendation(BaseModel):
    winner_asin: Optional[str]
    winner_title: Optional[str]
    reasoning: str
    runner_ups: List[str] = Field(default_factory=list)
    no_match_reason: Optional[str] = None


# ===========================================================================
# 3. GRAPH STATE
# ===========================================================================

class AgentState(TypedDict, total=False):
    raw_keyword: str
    intent: dict                 # QueryIntent.model_dump()
    candidates: List[dict]       # List[Product.model_dump()]
    survivors: List[dict]        # post budget/quality filter
    final: dict                  # FinalRecommendation.model_dump()
    retries_used: int
    error: Optional[str]
    status: Literal["running", "no_results", "done", "failed"]


# ===========================================================================
# 4. HELPERS
# ===========================================================================

def _ask_llm_json(system_prompt: str, user_prompt: str, schema: BaseModel):
    """Call Claude and force-parse the response into a pydantic schema, with one repair retry."""
    messages = [
        SystemMessage(content=system_prompt + "\n\nRespond with ONLY valid JSON. No markdown fences, no commentary."),
        HumanMessage(content=user_prompt),
    ]
    raw = llm.invoke(messages).content
    raw = raw.strip().strip("```json").strip("```").strip()
    try:
        return schema.model_validate_json(raw)
    except (ValidationError, json.JSONDecodeError) as e:
        log.warning(f"JSON parse failed, attempting repair: {e}")
        repair_msg = [
            SystemMessage(content="You returned invalid JSON. Return ONLY corrected valid JSON matching the requested schema, nothing else."),
            HumanMessage(content=f"Original bad output:\n{raw}\n\nFix it."),
        ]
        raw2 = llm.invoke(repair_msg).content.strip().strip("```json").strip("```").strip()
        return schema.model_validate_json(raw2)


@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=2, max=10),
    retry=retry_if_exception_type(requests.RequestException),
)
def _amazon_search(query: str, max_price: Optional[float], limit: int) -> List[dict]:
    """
    Real Amazon search via RapidAPI ("Real-Time Amazon Data" or similar).
    If no RAPIDAPI_KEY is configured, falls back to a deterministic MOCK so the
    graph is runnable end-to-end in Colab without any keys.
    """
    if not RAPIDAPI_KEY:
        log.warning("RAPIDAPI_KEY not set — using MOCK product data. Set the key for real results.")
        return _mock_search(query, max_price, limit)

    url = f"https://{RAPIDAPI_HOST}/search"
    headers = {"X-RapidAPI-Key": RAPIDAPI_KEY, "X-RapidAPI-Host": RAPIDAPI_HOST}
    params = {"query": query, "country": "US", "page": "1"}
    resp = requests.get(url, headers=headers, params=params, timeout=15)
    resp.raise_for_status()
    data = resp.json().get("data", {}).get("products", [])

    out = []
    for p in data[:limit]:
        price_raw = p.get("product_price", "").replace("$", "").replace(",", "")
        try:
            price = float(price_raw) if price_raw else None
        except ValueError:
            price = None
        if max_price and price and price > max_price:
            continue
        out.append({
            "asin": p.get("asin", ""),
            "title": p.get("product_title", ""),
            "price": price,
            "rating": p.get("product_star_rating") and float(p["product_star_rating"]),
            "review_count": p.get("product_num_ratings"),
            "url": p.get("product_url"),
            "thumbnail": p.get("product_photo"),
        })
    return out


def _mock_search(query: str, max_price: Optional[float], limit: int) -> List[dict]:
    """Deterministic fake data so the whole graph can be demoed without API keys."""
    base = [
        {"asin": "B0MOCK001", "title": f"{query.title()} Pro (Editor's Choice)", "price": 49.99, "rating": 4.6, "review_count": 8200},
        {"asin": "B0MOCK002", "title": f"{query.title()} Budget Edition", "price": 19.99, "rating": 4.1, "review_count": 1500},
        {"asin": "B0MOCK003", "title": f"{query.title()} Premium XL", "price": 89.99, "rating": 4.8, "review_count": 3100},
        {"asin": "B0MOCK004", "title": f"{query.title()} Lite", "price": 14.50, "rating": 3.6, "review_count": 420},
        {"asin": "B0MOCK005", "title": f"{query.title()} Standard", "price": 34.99, "rating": 4.3, "review_count": 5600},
    ]
    if max_price:
        base = [b for b in base if b["price"] <= max_price]
    return base[:limit]


def _fetch_reviews(asin: str, title: str) -> List[str]:
    """
    Real review pull would call the reviews endpoint of your Amazon API provider.
    Falls back to mock snippets when no key is set, keeping shape identical.
    """
    if not RAPIDAPI_KEY:
        return [
            f"Works as described, good value — bought for '{title[:30]}...' use case.",
            "Shipping was fast, packaging slightly damaged but product fine.",
            "Quality dropped compared to previous version, mixed feelings.",
        ]
    try:
        url = f"https://{RAPIDAPI_HOST}/product-reviews"
        headers = {"X-RapidAPI-Key": RAPIDAPI_KEY, "X-RapidAPI-Host": RAPIDAPI_HOST}
        resp = requests.get(url, headers=headers, params={"asin": asin, "country": "US"}, timeout=15)
        resp.raise_for_status()
        reviews = resp.json().get("data", {}).get("reviews", [])
        return [r.get("review_comment", "")[:300] for r in reviews[:5] if r.get("review_comment")]
    except requests.RequestException as e:
        log.warning(f"Review fetch failed for {asin}: {e}")
        return []


# ===========================================================================
# 5. AGENT NODES
# ===========================================================================

def query_interpreter_agent(state: AgentState) -> AgentState:
    log.info("[1/6] QueryInterpreterAgent")
    system = (
        "You are an e-commerce query interpreter. Given a long-tail customer search phrase, "
        "extract a clean search query plus structured shopping intent."
    )
    user = f'Long-tail keyword: "{state["raw_keyword"]}"\n\nSchema fields: core_query, max_budget_usd, min_rating, must_have_attributes, nice_to_have_attributes.'
    try:
        intent = _ask_llm_json(system, user, QueryIntent)
        return {**state, "intent": intent.model_dump(), "retries_used": 0, "status": "running"}
    except Exception as e:
        log.error(f"Intent parsing failed: {e}")
        return {**state, "error": str(e), "status": "failed"}


def product_search_agent(state: AgentState) -> AgentState:
    log.info("[2/6] ProductSearchAgent")
    intent = state["intent"]
    try:
        results = _amazon_search(intent["core_query"], intent.get("max_budget_usd"), MAX_CANDIDATES)
        if not results:
            return {**state, "candidates": [], "status": "no_results"}
        return {**state, "candidates": results, "status": "running"}
    except Exception as e:
        log.error(f"Search failed: {e}")
        return {**state, "error": str(e), "status": "failed"}


def review_mining_agent(state: AgentState) -> AgentState:
    log.info("[3/6] ReviewMiningAgent")
    candidates = state["candidates"]
    for c in candidates:
        c["review_snippets"] = _fetch_reviews(c["asin"], c["title"])
        time.sleep(0.1)  # gentle rate-limit pacing
    return {**state, "candidates": candidates}


def quality_scoring_agent(state: AgentState) -> AgentState:
    log.info("[4/6] QualityScoringAgent")
    candidates = state["candidates"]
    intent = state["intent"]
    system = (
        "You are a meticulous product-quality analyst. Score a single Amazon product from 0-10 "
        "based on its rating, review volume, review sentiment/content, and fit to the buyer's "
        "must-have attributes. Flag any red flags found in reviews (e.g. durability complaints, "
        "fake-review suspicion, frequent returns)."
    )
    for c in candidates:
        user = (
            f"Buyer must-haves: {intent.get('must_have_attributes')}\n"
            f"Product: {c['title']}\n"
            f"Price: {c.get('price')}\n"
            f"Rating: {c.get('rating')} ({c.get('review_count')} reviews)\n"
            f"Review snippets: {c.get('review_snippets')}\n\n"
            "Schema fields: quality_score, rationale, red_flags."
        )
        try:
            qa = _ask_llm_json(system, user, QualityAssessment)
            c["quality_score"] = qa.quality_score
            c["quality_rationale"] = qa.rationale
        except Exception as e:
            log.warning(f"Quality scoring failed for {c['asin']}: {e}")
            c["quality_score"] = (c.get("rating") or 0) * 2  # graceful fallback heuristic
            c["quality_rationale"] = "Fallback heuristic score (rating x2) — LLM scoring failed."
    return {**state, "candidates": candidates}


def budget_filter_agent(state: AgentState) -> AgentState:
    log.info("[5/6] BudgetFilterAgent")
    intent = state["intent"]
    max_budget = intent.get("max_budget_usd")
    min_rating = intent.get("min_rating", 4.0)
    candidates = state["candidates"]

    survivors = []
    for c in candidates:
        if max_budget and c.get("price") and c["price"] > max_budget:
            continue
        if c.get("rating") is not None and c["rating"] < min_rating - 0.3:  # small tolerance band
            continue
        if c.get("quality_score") is not None and c["quality_score"] < 4.0:
            continue
        survivors.append(c)

    if not survivors:
        return {**state, "survivors": [], "status": "no_results"}
    return {**state, "survivors": survivors, "status": "running"}


def retry_or_fail_agent(state: AgentState) -> AgentState:
    """Loosens the budget once and routes back to search; otherwise terminates with no-match."""
    retries = state.get("retries_used", 0)
    if retries < MAX_RETRY_LOOSEN:
        intent = state["intent"]
        if intent.get("max_budget_usd"):
            intent["max_budget_usd"] = round(intent["max_budget_usd"] * (1 + BUDGET_RELAX_PCT), 2)
            log.info(f"No survivors — relaxing budget to {intent['max_budget_usd']} and retrying.")
        return {**state, "intent": intent, "retries_used": retries + 1, "status": "running"}
    log.info("No survivors after retry budget exhausted — terminating with no-match.")
    return {
        **state,
        "final": FinalRecommendation(
            winner_asin=None,
            winner_title=None,
            reasoning="",
            no_match_reason="No product matched the budget, rating, and quality thresholds even after relaxing budget by "
                            f"{int(BUDGET_RELAX_PCT*100)}%. Consider raising the budget or relaxing must-have attributes.",
        ).model_dump(),
        "status": "failed",
    }


def decision_agent(state: AgentState) -> AgentState:
    log.info("[6/6] DecisionAgent")
    survivors = state["survivors"]
    intent = state["intent"]
    system = (
        "You are the final decision-maker for a product recommendation system. Given a shortlist "
        "of candidate products with quality scores, prices, ratings, and review evidence, pick the "
        "single best product for the buyer's stated needs, name 1-2 runner-ups, and explain your reasoning "
        "in 3-5 sentences referencing concrete evidence (price, rating, review content, quality rationale)."
    )
    user = (
        f"Buyer intent: {intent}\n\n"
        f"Shortlisted candidates:\n{json.dumps(survivors, indent=2)}\n\n"
        "Schema fields: winner_asin, winner_title, reasoning, runner_ups (list of asins), no_match_reason (null)."
    )
    try:
        decision = _ask_llm_json(system, user, FinalRecommendation)
        return {**state, "final": decision.model_dump(), "status": "done"}
    except Exception as e:
        log.error(f"Decision agent failed: {e}")
        # graceful fallback: highest quality_score wins
        best = max(survivors, key=lambda c: c.get("quality_score") or 0)
        fallback = FinalRecommendation(
            winner_asin=best["asin"],
            winner_title=best["title"],
            reasoning=f"Fallback ranking (LLM decision failed: {e}). Selected highest quality_score "
                      f"({best.get('quality_score')}) among survivors.",
            runner_ups=[c["asin"] for c in survivors if c["asin"] != best["asin"]][:2],
        )
        return {**state, "final": fallback.model_dump(), "status": "done"}


# ===========================================================================
# 6. CONDITIONAL ROUTING
# ===========================================================================

def route_after_search(state: AgentState) -> str:
    return "review_mining" if state["status"] == "running" else "retry_or_fail"


def route_after_budget_filter(state: AgentState) -> str:
    return "decision" if state["status"] == "running" else "retry_or_fail"


def route_after_retry(state: AgentState) -> str:
    return "product_search" if state["status"] == "running" else END


# ===========================================================================
# 7. BUILD GRAPH
# ===========================================================================

def build_graph():
    g = StateGraph(AgentState)

    g.add_node("query_interpreter", query_interpreter_agent)
    g.add_node("product_search", product_search_agent)
    g.add_node("review_mining", review_mining_agent)
    g.add_node("quality_scoring", quality_scoring_agent)
    g.add_node("budget_filter", budget_filter_agent)
    g.add_node("retry_or_fail", retry_or_fail_agent)
    g.add_node("decision", decision_agent)

    g.set_entry_point("query_interpreter")

    g.add_edge("query_interpreter", "product_search")
    g.add_conditional_edges("product_search", route_after_search, {
        "review_mining": "review_mining",
        "retry_or_fail": "retry_or_fail",
    })
    g.add_edge("review_mining", "quality_scoring")
    g.add_edge("quality_scoring", "budget_filter")
    g.add_conditional_edges("budget_filter", route_after_budget_filter, {
        "decision": "decision",
        "retry_or_fail": "retry_or_fail",
    })
    g.add_conditional_edges("retry_or_fail", route_after_retry, {
        "product_search": "product_search",
        END: END,
    })
    g.add_edge("decision", END)

    return g.compile()


# ===========================================================================
# 8. RUN
# ===========================================================================

def select_product(long_tail_keyword: str) -> dict:
    graph = build_graph()
    init_state: AgentState = {"raw_keyword": long_tail_keyword, "status": "running"}
    final_state = graph.invoke(init_state)
    return final_state


if __name__ == "__main__":
    # ----------------------- SETUP (Colab) -----------------------
    # %pip install -q langgraph langchain-anthropic langchain-core requests tenacity pydantic
    # # os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
    # # os.environ["RAPIDAPI_KEY"] = ""   # optional — leave blank to use mock data
    # ----------------------------------------------------------------

    keyword = "lightweight waterproof running shoes for flat feet under 70 dollars"
    result = select_product(keyword)

    print("\n================ FINAL RESULT ================")
    print(json.dumps(result.get("final"), indent=2))
    print("\n================ SURVIVOR SHORTLIST ================")
    print(json.dumps(result.get("survivors", []), indent=2))